## Creating Gold Dimension Tables Using Spark

Transforming silver tables into business-ready gold dimension tables using PySpark:
* **gld_dim_products** - Product dimension with brand and category enrichment
* **gld_dim_customers** - Customer dimension with regional mapping
* **gld_dim_date** - Date dimension with calendar attributes

**Key Features**:
- Spark broadcast joins for data shuffling and data enrichment
- Business logic transformations (regional mapping, date attributes)
- Optimized for analytics and BI workloads

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
catalog_name = dbutils.widgets.get("catalog_name")

print(f"Using catalog: {catalog_name}")

In [0]:
# Products dimension with brand and category enrichment

df_brands = spark.table(f"{catalog_name}.silver.slv_brands")
df_category = spark.table(f"{catalog_name}.silver.slv_category")
df_products = spark.table(f"{catalog_name}.silver.slv_products")

# Brand-category lookup using broadcast join
df_brands_categories = (df_brands
    .join(F.broadcast(df_category), df_brands.category_code == df_category.category_code, "inner")
    .select(
        df_brands.brand_name,
        df_brands.brand_code,
        df_category.category_name,
        df_category.category_code
    )
)

# Enrich products with brand and category details using broadcast join
df_dim_products = (df_products
    .join(F.broadcast(df_brands_categories), df_products.brand_code == df_brands_categories.brand_code, "left")
    .select(
        df_products.product_id,
        df_products.sku,
        df_products.category_code,
        F.coalesce(df_brands_categories.category_name, F.lit("Not Available")).alias("category_name"),
        df_products.brand_code,
        F.coalesce(df_brands_categories.brand_name, F.lit("Not Available")).alias("brand_name"),
        F.coalesce(df_products.color, F.lit("Unknown")).alias("color"),
        df_products.size,
        df_products.material,
        df_products.weight_grams,
        df_products.length_cm,
        df_products.width_cm,
        df_products.height_cm,
        df_products.rating_count
    )
)

# Write to gold
(df_dim_products
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.gold.gld_dim_products")
)

In [0]:
# Build customers dimension with regional mapping

df_customers = spark.table(f"{catalog_name}.silver.slv_customers")

# Add region based on country
df_dim_customers = (df_customers
    .withColumn("region",
        F.when(F.col("country") == "INDIA", "Asia")
         .when(F.col("country") == "AUSTRALIA", "Oceania")
         .when(F.col("country") == "UNITED KINGDOM", "Europe")
         .when(F.col("country") == "UNITED STATES", "North America")
         .when(F.col("country") == "UNITED ARAB EMIRATES", "Middle East")
         .when(F.col("country") == "SINGAPORE", "Asia")
         .when(F.col("country") == "CANADA", "North America")
         .otherwise("Other")
    )
    .select("customer_id", "phone", "country", "region")
)

# Write to gold
(df_dim_customers
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.gold.gld_dim_customers")
)

In [0]:
# Build date dimension with calendar attributes

df_calendar = spark.table(f"{catalog_name}.silver.slv_calendar")

# Add date attributes
df_dim_date = (df_calendar
    .withColumn("date_id", F.date_format(F.col("date"), "yyyyMMdd").cast("int"))
    .withColumn("month_name", F.date_format(F.col("date"), "MMMM"))
    .withColumn("is_weekend",
        F.when(F.col("day_name").isin(["Saturday", "Sunday"]), 1)
         .otherwise(0)
    )
    .select(
        "date_id",
        "date",
        "year",
        "month_name",
        "day_name",
        "is_weekend",
        "quarter",
        "week_of_year"
    )
)

# Write to gold
(df_dim_date
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")
)

In [0]:
print("Gold Dimension Tables Summary")

tables = [
    "gld_dim_products",
    "gld_dim_customers",
    "gld_dim_date"
]

for table in tables:
    count = spark.table(f"{catalog_name}.gold.{table}").count()
    print(f"{table:25} : {count:,} rows")

print("\n All gold dimension tables built successfully!")

In [0]:

# Quality check - Duplicate Validation
print("\n Data Quality Checks - Duplicate Validation")

# Check gld_dim_products
df_products_check = spark.table(f"{catalog_name}.gold.gld_dim_products")
total_products = df_products_check.count()
unique_products = df_products_check.select("product_id").distinct().count()
duplicates_products = total_products - unique_products

print(f"gld_dim_products:")
print(f"Total rows: {total_products:,}")
print(f"Unique product_ids: {unique_products:,}")
print(f"Duplicates: {duplicates_products}")

# Check gld_dim_customers
df_customers_check = spark.table(f"{catalog_name}.gold.gld_dim_customers")
total_customers = df_customers_check.count()
unique_customers = df_customers_check.select("customer_id").distinct().count()
duplicates_customers = total_customers - unique_customers

print(f"gld_dim_customers:")
print(f"Total rows: {total_customers:,}")
print(f"Unique customer_ids: {unique_customers:,}")
print(f"Duplicates: {duplicates_customers}")

# Check gld_dim_date
df_date_check = spark.table(f"{catalog_name}.gold.gld_dim_date")
total_dates = df_date_check.count()
unique_dates = df_date_check.select("date_id").distinct().count()
duplicates_dates = total_dates - unique_dates

print(f"gld_dim_date:")
print(f"Total rows: {total_dates:,}")
print(f"Unique date_ids: {unique_dates:,}")
print(f"Duplicates: {duplicates_dates}")

# Overall validation
if duplicates_products == 0 and duplicates_customers == 0 and duplicates_dates == 0:
    print("\n  Data Quality Check PASSED - No duplicates found!")
else:
    print("\n Data Quality Check FAILED - Duplicates detected!")